# 03 Conditional logic — on QCSS

Adapted from the official QICK demo [`03_Conditional_logic.ipynb`](https://github.com/openquantumhardware/qick/blob/9d98886/qick_demos/03_Conditional_logic.ipynb) (qick 0.2.370, MIT License, © 2021 Open Quantum Hardware).

Branch inside a tProcessor program on a register value (conditional jumps).

**What changes from the official demo:** the two cells marked `QCSS` — starting the simulated board, and connecting through `make_proxy` instead of running on the board. Every other cell is the upstream code, unchanged except where a `QCSS` comment says so.

**What this shows:** qick's own client library compiles, uploads and acquires exactly as it does against a board, and gets back data of the shape, dtype and scale a board returns. It does **not** prove the numbers are identical to any particular physical board: no real-board reference data is compared here.

In [ ]:
# QCSS: start the simulated ZCU216 board (tProc v1 firmware, as these demos expect)
import sys; sys.path.insert(0, '..')
import _lab, _proof
lab = _lab.start(qick=True, firmware='zcu216_v1_5q', loopback=('1:0',))

# <center> Conditional logic with the QICK demonstration

In this demo you will send and receive a pulse in loopback conditional on the value of a number written to the QICK. A similar application of the QICK conditional logic can enable the active reset of qubits. 

In [ ]:
# Import the QICK drivers and auxiliary libraries
from qick import *
%pylab inline

In [ ]:
# Load bitstream with custom overlay
# QCSS: the official demo runs on the board itself (soc = QickSoc()). From a PC you reach a board through
# QICK's Pyro4 server instead (qick.pyro.make_proxy) -- on a real board only ns_host would differ.
from qick.pyro import make_proxy
soc, soccfg = make_proxy(ns_host="127.0.0.1", ns_port=8888, proxy_name="myqick")
_lab.keep_errors_visible()  # QCSS: make_proxy hides cell errors from Jupyter; keep them failing


### Hardware Configuration

tProc channel 7   : DAC 229 CH3  <-> Readout channel 0 : ADC 224 CH0


In [ ]:
class LoopbackProgram(AveragerProgram):
    def initialize(self):
        cfg=self.cfg   

        # set the nyquist zone
        self.declare_gen(ch=cfg["res_ch"], nqz=1)

        #configure the readout lengths and downconversion frequencies
        self.declare_readout(ch=cfg["ro_ch"], length=self.cfg["readout_length"],
                             freq=self.cfg["pulse_freq"], gen_ch=cfg["res_ch"])

        freq=self.freq2reg(cfg["pulse_freq"], gen_ch=cfg["res_ch"], ro_ch=cfg["ro_ch"])  # convert frequency to dac frequency (ensuring it is an available adc frequency)
        self.set_pulse_registers(ch=cfg["res_ch"], style="const", freq=freq, phase=0, gain=cfg["pulse_gain"], 
                                 length=cfg["length"])
        
        # Set threshold
        self.regwi(0,1,self.cfg["threshold"])

        # Set value of number 
        self.regwi(0,2,self.cfg["number"])

        self.synci(200)  # give processor some time to configure pulses
    
    def body(self):
        self.trigger(adcs=[self.cfg["ro_ch"]],adc_trig_offset=self.cfg["adc_trig_offset"])  # trigger the adc acquisition
        
        # Apply conditional logic statement: When number is smaller than threshold, skip over playing the pulse.
        self.condj(0,2,'<',1,'LABEL') 
        self.pulse(ch=self.cfg["res_ch"]) # play readout pulse
        self.label('LABEL')

        # control should wait until the readout is over
        self.wait_all() 

        self.sync_all(self.us2cycles(self.cfg["relax_delay"]))  # sync all channels

### When the value of <code> number </code> is above the threshold, the conditional statement is False so you see a loopback pulse.

In [ ]:
config={"res_ch":1, # --Fixed  # QCSS bench cable: generator 1 -> readout 0 (on a real board: the DAC you cabled)
        "ro_ch":0, # --Fixed
        "reps":1, # --Fixed
        "relax_delay":1.0, # --Fixed
        "res_phase":0, # --Fixed
        "length":20, # [Clock ticks]        
        "readout_length":200, # [Clock ticks]
        "pulse_gain":1000, # [DAC units]
        "pulse_freq": 100, # [MHz]
        "adc_trig_offset": 100, # [Clock ticks]
        "soft_avgs":100,
        ### New variables
        "number": 100,
        "threshold": 50
       }

prog =LoopbackProgram(soccfg, config)
(iq0,) = prog.acquire_decimated(soc, progress=True)

In [ ]:
# Plot results.
plt.figure(1)
plt.plot(iq0[0], label="I value; ADC 0")
plt.plot(iq0[1], label="Q value; ADC 0")
plt.ylabel("a.u.")
plt.xlabel("Clock ticks")
plt.title("Averages = " + str(config["soft_avgs"]))
plt.legend()
plt.savefig("images/Conditional_False_Pulse.pdf", dpi=350)

### When the value of <code> number </code> is below the threshold, the conditional statement is True so you don't see a loopback pulse.

In [ ]:
config["number"]=10

prog =LoopbackProgram(soccfg, config)
(iq0,) = prog.acquire_decimated(soc, progress=True)

In [ ]:
# Plot results.
plt.figure(1)
plt.plot(iq0[0], label="I value; ADC 0")
plt.plot(iq0[1], label="Q value; ADC 0")
plt.ylabel("a.u.")
plt.xlabel("Clock ticks")
plt.title("Averages = " + str(config["soft_avgs"]))
plt.legend()
plt.savefig("images/Conditional_True_NoPulse.pdf", dpi=350)

## Does the client think it is a real board?

In [ ]:
_proof.qick_surface(soc, soccfg)

## Teardown

In [ ]:
lab.stop()